# Governed Agentic RAG — run on Google Colab (T4 GPU)

Run the cells **top to bottom**. Assumes you've already `git clone`d the repo into Colab.

**Before you start:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*.

What this does: install deps → install + start Ollama → pull `gemma4:12b` → build the
index → run the governance ablation (with RAGAS faithfulness, meaningful on the 12B judge)
→ show the metrics and plots.

## 1. Confirm the GPU (should show a Tesla T4)

In [ ]:
!nvidia-smi

## 2. Enter the repo
Change `REPO_DIR` if you cloned to a different path.

In [ ]:
import os
REPO_DIR = "/content/drive/MyDrive/governed-agentic-rag"   # <-- change if you cloned elsewhere
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
assert os.path.exists("scripts/build_index.py"), "Not in the repo root — fix REPO_DIR"

## 3. Install Python dependencies (Colab-specific)

Uses `requirements-colab.txt`, which **omits torch/numpy/pandas/matplotlib** (Colab already
has them — reinstalling torch can break the GPU) and **omits langgraph** (unused by the eval).
First we remove Colab's preinstalled `langgraph 1.x`, which otherwise conflicts with the
`langchain-core 0.3.x` that ragas needs.

Harmless leftover warnings about `requests`/`cryptography` versions can be ignored.

In [ ]:
# remove Colab's preinstalled langgraph 1.x (unused here, conflicts with langchain-core 0.3.x)
!pip uninstall -y -q langgraph langgraph-prebuilt langgraph-checkpoint langgraph-sdk
# install the Colab-tuned deps
!pip install -q -r requirements-colab.txt

## 4. Install Ollama, start the server, pull the model
`ollama serve` runs in the background; we wait a few seconds, then pull `gemma4:12b`
(~8 GB, fits the T4). This can take a few minutes the first time.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull gemma4:12b
!ollama list

## 5. Choose where files live + which model
**Recommended:** mount Google Drive so the index + results persist across sessions, and
select the `colab` profile (which sets Drive paths **and** `gemma4:12b`).

Env vars set here are inherited by the `!python ...` cells below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["GRAG_PROFILE"] = "colab"   # paths -> Drive, model -> gemma4:12b

# --- ephemeral alternative (no Drive; lost when the VM recycles) ---
# os.environ["GRAG_BASE_DIR"] = "/content/governed_rag"
# os.environ["GRAG_MODEL"]    = "gemma4:12b"

import sys; sys.path.insert(0, ".")
from src.config import load_config
cfg = load_config()
print("profile:", cfg["_profile"], "| model:", cfg["llm"]["model"])
print("qdrant :", cfg["paths"]["qdrant_dir"])
print("faithfulness backend:", cfg["evaluation"]["faithfulness_backend"])

## 6. Build the index (once)
On the T4 this is fast (GPU embedding). If you mounted Drive and already built it in a
previous session, `index_status.py` will show the count and you can **skip the build**.

In [ ]:
!python scripts/index_status.py
# If it says 'No collection ...', build it:
!python scripts/build_index.py

## 7. Run the governance ablation
6 configs × (boundary=5, poison=4, qa=8) with RAGAS faithfulness on the 12B judge.
Much faster than CPU. Bump `--n-boundary` to 10–20 for smoother numbers if you like.

In [ ]:
!python scripts/run_eval.py --n-boundary 5

## 8. View the results (metrics + plots)

In [ ]:
import json, os
from IPython.display import Image, display
from src.config import load_config
art = load_config()["paths"]["artifacts_dir"]
print(json.dumps(json.load(open(os.path.join(art, "metrics.json"))), indent=2))
display(Image(os.path.join(art, "tradeoff.png")))
display(Image(os.path.join(art, "safety_bars.png")))

## 9. (Optional) demo — same query, governance on vs off
Run the walkthrough notebook cells, or open `notebooks/demo.ipynb`.

**For your report:** copy the values from `metrics.json` into the tables in
`reports/report.md`, and cite the model as `gemma4:12b` (the one these numbers came from).